# Пример 12. Линейная алгебра: пересечение подпространств и CCA

## Тема

**Раздел книги:** Линейная алгебра.  
**Математическая тема:** подпространства, базис и размерность, пересечение двух подпространств $\mathbb{R}^n$ через ядро блочной матрицы $[V \mid -W]$, ранг и формула Грассмана $\dim(U_1+U_2)=\dim U_1+\dim U_2-\dim(U_1\cap U_2)$; связь с Canonical Correlation Analysis (CCA) и multi-view learning в ML.

## Условие
Даны два подпространства $\mathbb{R}^4$, заданные как линейные оболочки трёх векторов каждое:
$$
U_1=\operatorname{span}\!\left\{\begin{bmatrix}1\\1\\-3\\1\end{bmatrix},\begin{bmatrix}2\\-1\\0\\-1\end{bmatrix},\begin{bmatrix}-1\\1\\-1\\1\end{bmatrix}\right\},\quad
U_2=\operatorname{span}\!\left\{\begin{bmatrix}-1\\-2\\2\\1\end{bmatrix},\begin{bmatrix}2\\-2\\0\\0\end{bmatrix},\begin{bmatrix}-3\\6\\-2\\-1\end{bmatrix}\right\}.
$$
Требуется определить базис пересечения $U_1\cap U_2$.

## Математическая идея
Вектор $x$ лежит в пересечении тогда и только тогда, когда он представим **с обеих сторон**:
$$
x = a_1 v_1+a_2 v_2+a_3 v_3 = b_1 w_1+b_2 w_2+b_3 w_3.
$$
Перенося всё в одну сторону, получаем однородную систему
$$
a_1 v_1+a_2 v_2+a_3 v_3 - b_1 w_1-b_2 w_2-b_3 w_3 = 0,
$$
то есть нужно найти **ядро** блочной матрицы $M=[\,V\mid -W\,]\in\mathbb{R}^{4\times 6}$. Каждое решение $(a,b)$ даёт вектор $x=Va$; множество таких $x$ и есть $U_1\cap U_2$, а его базис — линейно независимые полученные векторы $x$. Полезный контроль размерности: $\dim(U_1+U_2)=\dim U_1+\dim U_2-\dim(U_1\cap U_2)$.

## Решение

**Шаг 1. Замкнутость образующих.** Внутри каждого набора есть зависимость: $v_1=2v_2+3v_3$ и $w_1=-2w_2-w_3$, поэтому $\dim U_1=\dim U_2=2$ (базисы — $\{v_2,v_3\}$ и $\{w_2,w_3\}$).

**Шаг 2. Блочная система.** Матрица коэффициентов при $(a_1,a_2,a_3,b_1,b_2,b_3)$:
$$
M=[V\mid -W]=\begin{bmatrix}
1&2&-1&1&-2&3\\
1&-1&1&2&2&-6\\
-3&0&-1&-2&0&2\\
1&-1&1&-1&0&-1
\end{bmatrix}.
$$
Её ядро трёхмерно (ранг $=3$): два направления отвечают за внутренние зависимости $v_1-2v_2-3v_3=0$ и $w_1+2w_2+w_3=0$, а третье — за само пересечение.

**Шаг 3. Вектор пересечения.** Нетривиальное решение $a=(0,6,4)$, $b=(0,-7,-2)$ даёт
$$
x = 6v_2+4v_3 = \begin{bmatrix}8\\-2\\-4\\-2\end{bmatrix} = 3.5\,w_2+1\,w_3,
$$
то есть после деления на общий множитель $x=(4,-1,-2,-1)^\top$.

**Шаг 4. Размерность.** Объединение $\{v_2,v_3,w_2,w_3\}$ имеет ранг $3$ (например, $w_2\notin U_1$), значит $\dim(U_1+U_2)=3$ и $\dim(U_1\cap U_2)=2+2-3=1$.

**Ответ:** базис пересечения — одно направление
$$
\left\{\begin{bmatrix}4\\-1\\-2\\-1\end{bmatrix}\right\}.
$$

In [2]:
import numpy as np

# Столбцы-образующие подпространств U1 и U2
V = np.array([[1, 2, -1],
             [1, -1, 1],
             [-3, 0, -1],
             [1, -1, 1]], dtype=float)           # U1
W = np.array([[-1, 2, -3],
             [-2, -2, 6],
             [2, 0, -2],
             [1, 0, -1]], dtype=float)           # U2

# Блочная матрица [V | -W] размером 4x6
M = np.hstack([V, -W])

# Корректное получение ядра через SVD
_, s, Vh = np.linalg.svd(M)                      # s имеет длину 4, Vh — форму (6, 6)
rank_M = np.linalg.matrix_rank(M, tol=1e-10)     # ранг = 3
nullspace = Vh[rank_M:]                          # строки с индекса rank_M — базис ядра
print(f"rank(M) = {rank_M}, dim ker(M) = {nullspace.shape[0]}")

# Каждое решение (a, b) даёт кандидата x = V @ a в пересечение
X = np.array([V @ n[:3] for n in nullspace]).T   # столбцы — кандидаты

# Оставляем линейно независимые столбцы — это и есть базис U1 ∩ U2
_, sx, _ = np.linalg.svd(X, full_matrices=False)
basis = X[:, sx > 1e-8]

print("dim(U1 ∩ U2) =", basis.shape[1])
print("Базисный вектор (столбец):\n", basis)

# Встроенная проверка принадлежности через ранг
x = basis[:, 0]
in_U1 = np.linalg.matrix_rank(np.column_stack([V, x])) == np.linalg.matrix_rank(V)
in_U2 = np.linalg.matrix_rank(np.column_stack([W, x])) == np.linalg.matrix_rank(W)
print(f"x ∈ U1: {in_U1} | x ∈ U2: {in_U2}")

rank(M) = 3, dim ker(M) = 3
dim(U1 ∩ U2) = 1
Базисный вектор (столбец):
 [[ 1.40601543]
 [-0.35150386]
 [-0.70300772]
 [-0.35150386]]
x ∈ U1: True | x ∈ U2: True


## Дополнительный пример: пересечение подпространств и CCA

**Идея.** В задаче 2.12 мы искали вектор, лежащий одновременно в $U_1$ и $U_2$, — то есть **общее направление** двух подпространств. В машинном обучении та же логика появляется в **Canonical Correlation Analysis (CCA)**: пусть $X\in\mathbb{R}^{n\times p}$ и $Y\in\mathbb{R}^{n\times q}$ — два представления одних и тех же объектов. CCA ищет направления $u,v$, максимизирующие корреляцию проекций
$$
\rho_1=\max_{u,v}\ \frac{u^\top \Sigma_{XY} v}{\sqrt{u^\top \Sigma_{XX} u}\ \sqrt{v^\top \Sigma_{YY} v}},
$$
что сводится к обобщённой задаче на собственные значения $\Sigma_{XX}^{-1}\Sigma_{XY}\Sigma_{YY}^{-1}\Sigma_{YX}u=\rho^2 u$. Число ненулевых канонических корреляций играет роль размерности пересечения $\dim(U_1\cap U_2)$: если $X$ и $Y$ порождены одним скрытым фактором, общая «линия» ровно одна, а шумовые признаки дают корреляции $\approx 0$.

In [3]:
import numpy as np
from sklearn.cross_decomposition import CCA

rng = np.random.default_rng(42)
n = 500

# Один общий скрытый фактор s — источник "пересечения" представлений
s = rng.normal(size=(n, 1))

# Два представления (views): линейно зависят от s + шум
A1 = rng.normal(size=(1, 4))          # 4 признака в X
A2 = rng.normal(size=(1, 4))          # 4 признака в Y
X = s @ A1 + rng.normal(scale=0.3, size=(n, 4))
Y = s @ A2 + rng.normal(scale=0.3, size=(n, 4))

# CCA: ищем общие направления (аналог U1 ∩ U2)
cca = CCA(n_components=2)
X_c, Y_c = cca.fit_transform(X, Y)

# Канонические корреляции = корреляции пар проекций покомпонентно
canon_corr = np.array([np.corrcoef(X_c[:, k], Y_c[:, k])[0, 1]
                       for k in range(X_c.shape[1])])

print("Канонические корреляции:", np.round(canon_corr, 3))
# Ожидаем: первая ~высокая (общее направление), вторая ~0 (шум)

Канонические корреляции: [0.988 0.075]


## Проверка результата

В первой части проверка выполняется через ядро блочной матрицы $[V\mid -W]$: вектор, полученный из ненулевого решения однородной системы, подставляется обратно в оба набора образующих, и его принадлежность $U_1$ и $U_2$ подтверждается равенством рангов $\operatorname{rank}[V\mid x]=\operatorname{rank}V$ и $\operatorname{rank}[W\mid x]=\operatorname{rank}W$; размерность пересечения дополнительно контролируется формулой $\dim(U_1+U_2)=\dim U_1+\dim U_2-\dim(U_1\cap U_2)$. Во второй части проверяется согласованность двух независимых реализаций одной и той же идеи: число канонических корреляций, существенно отличных от нуля, должно совпадать с размерностью общего подпространства, найденной в задаче 2.12 (в эксперименте заложено одно общее направление, поэтому высокая корреляция ровно одна), а обратный контроль — замена $Y$ на независимый шум — должен обнулить все корреляции, что соответствует вырожденному пересечению $\{0\}$.

## Вывод

Пересечение подпространств вычисляется через ядро блочной матрицы $[V\mid -W]$, а его размерность удобно контролировать формулой Грассмана; продемонстрированы ключевые идеи главы — линейная зависимость образующих, ранг, ядро и проекция решения обратно в исходное пространство. В машинном обучении этот аппарат является фундаментом методов поиска общих структур между несколькими представлениями данных: канонические направления CCA выступают статистическим аналогом векторов из $U_1\cap U_2$, а число ненулевых канонических корреляций — аналогом размерности пересечения. Понимание этой связи позволяет осознанно применять multi-view обучение, матричную факторизацию и методы сжатия, где «общая часть» представлений и есть носитель полезного сигнала.